# Week 07: More NMT

## Libraries Used

In [11]:
!pip install transformers datasets evaluate sentencepiece sacrebleu --upgrade


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

from random import random
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
from transformers import MarianTokenizer, MarianMTModel
import evaluate
import random

from collections import Counter

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Lecture 1: Vocbulary

### Question: What is the main limitation of using a fixed vocabulary in Neural Machine Translation?

The main limitation of using a fixed vocabulary in Neural Machine Translation (NMT) is its inability to handle rare or unseen words effectively. Unlike natural language, which is dynamic and constantly evolving, a fixed vocabulary assumes a static set of known words based on the training corpus. This leads to several issues:

- **Out-of-vocabulary (OOV) words**: Words not seen during training are replaced with a generic unknown token (e.g., `<UNK>`), causing loss of critical information during translation.
- **Named entities and neologisms**: These often appear infrequently and are therefore excluded from the fixed vocabulary, despite their importance to meaning.
- **Compounding and morphology**: In languages like German, where compounding is common, many long and unique word forms may not be present in the fixed vocabulary.
- **Scalability limits**: Expanding the vocabulary to include more words increases model size and training time, making it computationally expensive.

Overall, a fixed vocabulary restricts the model’s ability to generalize and accurately translate diverse and evolving real-world text.


### Question: Why is the use of an <UNK> token often considered insufficient for translating rare words?

The use of an unknown (`<UNK>`) token is often considered insufficient for translating rare words because it causes a loss of essential information in the translation. When a word is not found in the model’s fixed vocabulary, it is replaced with `<UNK>`, and the model can no longer represent its meaning or produce an accurate translation.

Key limitations include:

- **Loss of semantic content**: Rare or domain-specific words (like proper nouns, technical terms, or neologisms) often carry crucial meaning. Replacing them with `<UNK>` removes this meaning entirely from the output.
- **Poor user experience**: Translations with `<UNK>` tokens are incomplete or confusing to end users, especially if the dropped words are names, locations, or key phrases.
- **Inability to generalize**: The model treats all unseen words the same, ignoring any morphological or contextual clues that could help infer their role or meaning.
- **Breakdown in alignment**: When the source word is replaced by `<UNK>`, alignment between source and target becomes ambiguous, making post-processing or error analysis more difficult.

Overall, while `<UNK>` tokens allow the model to process sentences with unseen words, they result in degraded translation quality and are only a temporary workaround rather than a robust solution.


### Question:  What is the copy mechanism and how does the copy mechanism use attention weights to handle unknown words during translation?

The copy mechanism is a technique used in Neural Machine Translation (NMT) to handle rare or unknown words by leveraging the model's attention weights to align and substitute these words in the output.

Here's how it works:

- During training and inference, rare words that fall outside the model's fixed vocabulary are replaced with a special token (e.g., `<UNK>`).
- When generating the translation, the model uses its attention mechanism to determine which source words are most relevant for producing each target word.
- For each `<UNK>` token in the output, the attention weights indicate which source word the model was focusing on at that decoding step.
- The system then "copies" the most-attended source word—typically the unknown word itself—into the translated sentence.
- To improve quality, this copied word can optionally be passed through an external translation tool (e.g., a phrase table from an SMT system) before being inserted into the target sentence.

This approach enables the system to:
- Retain named entities, numbers, and rare terms that are otherwise lost with `<UNK>` replacement.
- Produce more semantically faithful translations.
- Avoid the need for full coverage of all possible vocabulary items in the training set.


### Quick Python Example

In [5]:
# Simulated source sentence with a rare word
source_tokens = ["I", "live", "in", "Cadswer"]  # "Cadswer" is rare/unseen

# Simulated target sentence with an <UNK> placeholder
target_tokens = ["Ich", "wohne", "in", "<UNK>"]

# Simulated attention weights (4 target tokens × 4 source tokens)
# Each row corresponds to attention weights from one target token
import numpy as np
attention_matrix = np.array([
    [0.6, 0.3, 0.1, 0.0],  # "Ich" attends to "I"
    [0.1, 0.7, 0.2, 0.0],  # "wohne" attends to "live"
    [0.0, 0.2, 0.7, 0.1],  # "in" attends to "in"
    [0.0, 0.1, 0.2, 0.7]   # "<UNK>" attends to "Cadswer"
])

# Apply copy mechanism
def apply_copy_mechanism(source, target, attn_matrix):
    copied_output = []
    for i, word in enumerate(target):
        if word == "<UNK>":
            # Find the source token with highest attention for this target token
            source_idx = np.argmax(attn_matrix[i])
            copied_output.append(source[source_idx])
        else:
            copied_output.append(word)
    return copied_output

copied_translation = apply_copy_mechanism(source_tokens, target_tokens, attention_matrix)
print("Final Translation:", " ".join(copied_translation))


Final Translation: Ich wohne in Cadswer


### Question: What are the main advantages and disadvantages of Byte Pair Encoding (BPE) in NMT?

Byte Pair Encoding (BPE) is a subword tokenization technique that helps address the open vocabulary problem in Neural Machine Translation (NMT) by breaking words into frequent subword units. This allows models to generalize better to rare and unseen words.

**Advantages:**
- **Open vocabulary handling**: BPE allows the model to represent any word, including rare or unseen ones, as a sequence of subword units.
- **Vocabulary compression**: By using subword units instead of whole words, BPE reduces the size of the vocabulary while maintaining high coverage.
- **Improved generalization**: The model can learn meaningful patterns across word fragments (e.g., prefixes, suffixes), improving performance on morphologically rich languages.
- **Compatibility with neural architectures**: BPE produces shorter sequences than character-level models, making it computationally more efficient.

**Disadvantages:**
- **Arbitrary splits**: BPE merges character pairs based on frequency, not linguistic boundaries, leading to subword units that may not be semantically meaningful.
- **Longer sequences**: Compared to word-level models, BPE still increases the number of tokens, which can raise computational cost and complexity.
- **Inconsistent splits**: Rare or out-of-domain words may be split into many small units, making it harder for the model to learn good representations.
- **Fixed merges**: Once trained, the merge rules are static and may not adapt well to domain shifts or evolving language.

Despite these trade-offs, BPE remains a widely adopted and effective approach for subword modeling in modern NMT systems.


### Example: BPE in practice

In [6]:
from collections import Counter, defaultdict

# Starting word list (can be sentences too)
vocab = {
    "low": 5,
    "lowest": 2,
    "newest": 6,
    "widest": 3
}

# Split each word into characters + end-of-word marker
def split_vocab(vocab):
    return {tuple(word) + ('</w>',): freq for word, freq in vocab.items()}

# Get frequency of all symbol pairs
def get_pair_freq(tokenized_vocab):
    pairs = Counter()
    for word, freq in tokenized_vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            pairs[pair] += freq
    return pairs

# Merge the most frequent pair
def merge_pair(pair_to_merge, tokenized_vocab):
    new_vocab = {}
    bigram = ' '.join(pair_to_merge)
    replacement = ''.join(pair_to_merge)
    for word, freq in tokenized_vocab.items():
        word_str = ' '.join(word)
        new_word_str = word_str.replace(bigram, replacement)
        new_vocab[tuple(new_word_str.split())] = freq
    return new_vocab

# Run BPE for a few iterations
tokenized_vocab = split_vocab(vocab)
num_merges = 10

for i in range(num_merges):
    pairs = get_pair_freq(tokenized_vocab)
    if not pairs:
        break
    most_freq = pairs.most_common(1)[0][0]
    tokenized_vocab = merge_pair(most_freq, tokenized_vocab)
    print(f"Step {i+1}: Merged {most_freq}")
    for word in tokenized_vocab:
        print("   ", word)


Step 1: Merged ('e', 's')
    ('l', 'o', 'w', '</w>')
    ('l', 'o', 'w', 'es', 't', '</w>')
    ('n', 'e', 'w', 'es', 't', '</w>')
    ('w', 'i', 'd', 'es', 't', '</w>')
Step 2: Merged ('es', 't')
    ('l', 'o', 'w', '</w>')
    ('l', 'o', 'w', 'est', '</w>')
    ('n', 'e', 'w', 'est', '</w>')
    ('w', 'i', 'd', 'est', '</w>')
Step 3: Merged ('est', '</w>')
    ('l', 'o', 'w', '</w>')
    ('l', 'o', 'w', 'est</w>')
    ('n', 'e', 'w', 'est</w>')
    ('w', 'i', 'd', 'est</w>')
Step 4: Merged ('w', 'est</w>')
    ('l', 'o', 'w', '</w>')
    ('l', 'o', 'west</w>')
    ('n', 'e', 'west</w>')
    ('w', 'i', 'd', 'est</w>')
Step 5: Merged ('l', 'o')
    ('lo', 'w', '</w>')
    ('lo', 'west</w>')
    ('n', 'e', 'west</w>')
    ('w', 'i', 'd', 'est</w>')
Step 6: Merged ('n', 'e')
    ('lo', 'w', '</w>')
    ('lo', 'west</w>')
    ('ne', 'west</w>')
    ('w', 'i', 'd', 'est</w>')
Step 7: Merged ('ne', 'west</w>')
    ('lo', 'w', '</w>')
    ('lo', 'west</w>')
    ('newest</w>',)
    ('w', 'i'

Byte Pair Encoding (BPE) is a data-driven algorithm that iteratively merges the most frequent adjacent pairs of symbols (typically characters or subword units) in a vocabulary to create new tokens. This process allows the model to build a compact and expressive subword vocabulary capable of representing any word.

Here’s how the toy BPE example in Python works:

- **Initialization**: Each word in the vocabulary is split into individual characters with a special end-of-word token (`</w>`). This helps the model differentiate word boundaries.

- **Counting pairs**: The algorithm scans the tokenized vocabulary to find the most frequent adjacent character pairs across all words, weighted by their frequency.

- **Merging**: The most frequent pair (e.g., `('e', 's')`) is merged into a single new symbol (e.g., `'es'`). This process is repeated for a fixed number of steps or until no frequent pairs remain.

- **Iteration**: Over multiple iterations, common patterns (like prefixes, suffixes, or root stems) are merged into longer subword units. For example, "newest" might eventually become `('new', 'est', '</w>')`.

- **Result**: The final vocabulary contains a mix of whole words and subword fragments that occur frequently in the data. This allows rare or unseen words to be decomposed into known subword units at inference time.

This simple example demonstrates the core logic of BPE without relying on large corpora or external libraries. In practice, BPE enables neural models to generalize better to morphologically rich or noisy input by avoiding the need for a fixed word-level vocabulary.


### Question: How does character-level NMT differ architecturally from word-based models?

Character-level Neural Machine Translation (NMT) differs from word-based models in both input representation and architectural design. Instead of using words or subword units as the basic input/output tokens, character-level models operate directly on sequences of individual characters.

**Key architectural differences:**

- **Input granularity**:
  - *Word-based models* represent sentences as sequences of word embeddings.
  - *Character-level models* represent sentences as sequences of character embeddings, treating each character (including spaces and punctuation) as a separate input.

- **Sequence length**:
  - Character-level sequences are much longer than word-based ones, leading to increased computational complexity and memory usage.
  - This requires models to be more robust in handling long-term dependencies.

- **Embedding strategy**:
  - Character-level models may use **convolutional** or **recurrent** layers to learn fixed-length representations (embeddings) from short character n-grams or sliding windows.
  - These embeddings are then fed into standard encoder-decoder architectures with attention.

- **Segmentation independence**:
  - Word-based models rely on tokenization, which may be language-dependent or error-prone.
  - Character-level models avoid tokenization entirely, which is particularly advantageous for languages with no clear word boundaries (e.g., Chinese or Thai).

- **Output generation**:
  - The decoder in character-level NMT generates one character at a time, making it slower than word-based decoding, but more flexible and expressive for generating novel or morphologically complex words.

Overall, character-level NMT provides greater robustness and flexibility at the cost of increased training and inference time. It is especially useful for handling noisy input, rare words, and morphologically rich languages.


### Question: What is the role of external tools (like phrase tables from SMT systems) in the copy mechanism?

The copy mechanism is a technique used in Neural Machine Translation (NMT) to handle rare or out-of-vocabulary (OOV) words by identifying and reusing specific words from the source sentence in the target sentence. Rather than attempting to generate a translation for an unknown word, the model "copies" it directly into the output, guided by attention weights.

**How it works:**
- During translation, rare source words are replaced by a placeholder token (e.g., `<UNK>`).
- The model uses **attention weights** to determine which source word the `<UNK>` token aligns with most strongly during decoding.
- The aligned source word is then copied into the target output, typically preserving its form (as with names or technical terms).

**Role of external tools:**
- In some cases, simply copying the source word is not sufficient (e.g., when a translation exists for the rare word but it's not in the model's vocabulary).
- To improve translation quality, the copy mechanism can integrate **external tools**, such as **phrase tables from Statistical Machine Translation (SMT)** systems, to translate the unknown word.
- After identifying the aligned source word via attention, the system queries the SMT phrase table to obtain a probable translation and inserts it into the target sentence in place of the `<UNK>` token.

**Benefits of using external tools:**
- Enables meaningful translations for rare or unseen words without retraining the NMT model.
- Preserves semantic content that would otherwise be lost with `<UNK>`.
- Combines strengths of SMT (robust phrase lookup) with NMT (contextual fluency), improving hybrid translation performance.

However, this hybrid method can introduce inconsistencies if the external translation is not well-aligned with the surrounding neural-generated text.


### Question: In what scenarios might BPE fail to produce meaningful subword units? Provide examples.

Byte Pair Encoding (BPE) can fail to produce meaningful subword units in scenarios where its frequency-based merging strategy does not align with linguistic boundaries or semantic coherence. Because BPE merges the most frequent adjacent symbol pairs without considering morphology or word meaning, the resulting units may be arbitrary or misleading.

**Scenarios where BPE may fail:**

- **Rare or domain-specific vocabulary**:
  - BPE may split technical or low-frequency words into subwords that are not semantically informative.
  - *Example*: The word `"cytokinesis"` might be broken into `"cy"`, `"tok"`, `"ine"`, `"sis"`, none of which reflect the biological meaning.

- **Named entities and proper nouns**:
  - Names often have idiosyncratic spellings and are unlikely to be frequent in the training corpus.
  - *Example*: `"Schwarzenegger"` could be split into `"Sch"`, `"war"`, `"zen"`, `"egger"`, which obscures the identity of the name.

- **Compounded or agglutinative words**:
  - In languages like German or Finnish, BPE may not respect natural morpheme boundaries.
  - *Example*: `"Donaudampfschifffahrtsgesellschaftskapitän"` might be split in ways that lose the internal structure, such as `"Donau"`, `"dampf"`, `"schiff"`, `"fahr"`, `"tsg"`, `"esellschaft"`, `"skapitän"`.

- **Over-fragmentation of unseen words**:
  - Unseen or very rare words might be split into many small units, which increases sequence length and makes modeling harder.
  - *Example*: `"quarkonium"` might be split into `["q", "ua", "rk", "on", "ium"]`, none of which carry useful sub-meaning.

- **Incorrect assumptions from frequent substrings**:
  - BPE might prioritize frequent substrings that occur across unrelated words, leading to subword units that group together coincidental character sequences.
  - *Example*: `"interview"` and `"intervene"` may cause `"inter"` to be merged, even though it's not a meaningful morpheme in `"intervene"`.

In all these cases, BPE's lack of linguistic awareness can lead to suboptimal tokenizations that hinder the model’s ability to learn meaningful representations.


### Question: Imagine you are designing an NMT system for a domain with frequent neologisms (e.g., social media). Which vocabulary handling approach would you choose and why?

For a Neural Machine Translation (NMT) system targeting a domain with frequent neologisms—such as social media, online forums, or meme culture—the most effective vocabulary handling approach would be **Byte Pair Encoding (BPE)** or a similar subword-based method (e.g., SentencePiece with Unigram LM), possibly in combination with a **copy mechanism**.

**Reasons for choosing BPE:**

- **Robustness to new words**: BPE can tokenize any word into known subword units, even if the word itself is unseen during training. This is ideal for handling constantly evolving vocabulary and slang.
- **Efficient vocabulary size**: By limiting the total number of subword tokens (e.g., 16K–32K), BPE ensures memory efficiency and avoids the overhead of full character-level models.
- **Generalization**: BPE allows the model to recognize and leverage common roots or affixes across neologisms (e.g., `"microblogging"`, `"doomscrolling"`), improving learning and fluency.
- **Improved performance over `<UNK>`**: Unlike word-based models that discard unknown words or replace them with an `<UNK>` token, BPE enables partial understanding and generation of new terms.

**Optional enhancement with the copy mechanism:**
- In cases where proper nouns, hashtags, or product names (e.g., `#Yassification`, `TikToker123`) should be preserved, integrating a copy mechanism ensures such tokens are carried over intact based on attention alignments.

This hybrid approach balances **coverage**, **efficiency**, and **fidelity**, making it well-suited to fast-changing linguistic environments like social media.


### Question: If a model consistently mistranslates named entities, what are two practical strategies you could use to address this?

If a model consistently mistranslates named entities (e.g., personal names, locations, organizations), this often indicates limitations in vocabulary coverage or alignment mechanisms. Two practical strategies to address this are:

1. **Integrate a Copy Mechanism**:
   - Use attention weights to identify source tokens that are likely named entities and directly copy them into the target output.
   - This is especially effective for transliteration or when the named entity should remain unchanged (e.g., "John Smith" in English and German).
   - Helps avoid `<UNK>` token substitutions and preserves identity-specific information.

2. **Use Named Entity Recognition (NER) with Post-Processing**:
   - Apply an NER tool to detect named entities in the source sentence before translation.
   - During or after translation, ensure these entities are either:
     - Copied directly into the target (preservation), or
     - Replaced with their known translations using a lookup table or dictionary.
   - This hybrid approach allows the NMT system to focus on translating the sentence structure while ensuring named entities are handled accurately.

Together, these strategies improve translation fidelity for high-value tokens and are particularly useful in formal, domain-specific, or multilingual settings where entity accuracy is critical.


### Example: In depth 

In [19]:
# Install dependencies first if needed
# pip install transformers datasets evaluate sacrebleu sentencepiece torch tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    MarianTokenizer, MarianMTModel, DataCollatorForSeq2Seq
)
import evaluate
from tqdm.auto import tqdm
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === Load WMT14 English-German ===
dataset = load_dataset("wmt14", "de-en", split="train[:1%]")
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_data = dataset['train'].select(range(200))
test_data = dataset['test'].select(range(50))

### === Part 1: BPE-Based MarianMT Model (Commented Out) ===
"""
bpe_model_name = "Helsinki-NLP/opus-mt-en-de"
bpe_tokenizer = MarianTokenizer.from_pretrained(bpe_model_name)
bpe_model = MarianMTModel.from_pretrained(bpe_model_name).to(device)
bpe_collator = DataCollatorForSeq2Seq(tokenizer=bpe_tokenizer, model=bpe_model)

def tokenize_bpe(example):
    source = example["translation"]["en"]
    target = example["translation"]["de"]
    inputs = bpe_tokenizer(source, truncation=True, padding="max_length", max_length=64)
    labels = bpe_tokenizer(target, truncation=True, padding="max_length", max_length=64)["input_ids"]
    inputs["labels"] = labels
    return inputs

train_bpe = train_data.map(tokenize_bpe)
test_bpe = test_data.map(tokenize_bpe)
train_bpe.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_bpe.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
train_loader_bpe = DataLoader(train_bpe, batch_size=4, shuffle=True, collate_fn=bpe_collator)
test_loader_bpe = DataLoader(test_bpe, batch_size=4, collate_fn=bpe_collator)
"""

### === Part 2: Non-BPE (Whitespace Tokenizer) ===
def whitespace_tokenize(example):
    example["src_tokens"] = example["translation"]["en"].split()
    example["tgt_tokens"] = example["translation"]["de"].split()
    return example

train_ws = train_data.map(whitespace_tokenize)
test_ws = test_data.map(whitespace_tokenize)

def build_vocab(data, key):
    counter = Counter()
    for ex in data:
        counter.update(ex[key])
    vocab = {tok: i+2 for i, tok in enumerate(counter)}  # Reserve 0=PAD, 1=UNK
    vocab["<pad>"] = 0
    vocab["<unk>"] = 1
    return vocab

src_vocab = build_vocab(train_ws, "src_tokens")
tgt_vocab = build_vocab(train_ws, "tgt_tokens")
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

def tokens_to_ids(example, src_vocab, tgt_vocab, max_length=64):
    src_ids = [src_vocab.get(t, src_vocab["<unk>"]) for t in example["src_tokens"][:max_length]]
    tgt_ids = [tgt_vocab.get(t, tgt_vocab["<unk>"]) for t in example["tgt_tokens"][:max_length]]
    src_ids += [src_vocab["<pad>"]] * (max_length - len(src_ids))
    tgt_ids += [tgt_vocab["<pad>"]] * (max_length - len(tgt_ids))
    return {"input_ids": torch.tensor(src_ids), "labels": torch.tensor(tgt_ids)}

train_ws = train_ws.map(lambda x: tokens_to_ids(x, src_vocab, tgt_vocab))
test_ws = test_ws.map(lambda x: tokens_to_ids(x, src_vocab, tgt_vocab))
train_ws.set_format(type='torch', columns=['input_ids', 'labels'])
test_ws.set_format(type='torch', columns=['input_ids', 'labels'])
train_loader_ws = DataLoader(train_ws, batch_size=4, shuffle=True)
test_loader_ws = DataLoader(test_ws, batch_size=4)

# === Tiny Transformer Model for Non-BPE
class TinySeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=256, nhead=4, num_layers=2):
        super().__init__()
        self.embedding_src = nn.Embedding(src_vocab_size, emb_dim)
        self.embedding_tgt = nn.Embedding(tgt_vocab_size, emb_dim)
        self.transformer = nn.Transformer(
            d_model=emb_dim, nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers
        )
        self.out = nn.Linear(emb_dim, tgt_vocab_size)

    def forward(self, src, tgt):
        src_emb = self.embedding_src(src).permute(1, 0, 2)
        tgt_emb = self.embedding_tgt(tgt).permute(1, 0, 2)
        memory = self.transformer(src_emb, tgt_emb)
        return self.out(memory).permute(1, 0, 2)

model_ws = TinySeq2Seq(len(src_vocab), len(tgt_vocab)).to(device)
optimizer_ws = torch.optim.Adam(model_ws.parameters(), lr=2e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

# === Train Non-BPE Model for 8 Epochs ===
EPOCHS = 8
for epoch in range(EPOCHS):
    model_ws.train()
    pbar = tqdm(train_loader_ws, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        src = batch["input_ids"].to(device)
        tgt = batch["labels"].to(device)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        logits = model_ws(src, tgt_in)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer_ws.zero_grad()
        loss.backward()
        optimizer_ws.step()
        pbar.set_postfix(loss=loss.item())

# === BLEU Evaluation for Non-BPE ===
bleu = evaluate.load("sacrebleu")

def evaluate_bleu_ws():
    model_ws.eval()
    preds, refs = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader_ws, desc="Eval Non-BPE"):
            src = batch["input_ids"].to(device)
            memory = model_ws.embedding_src(src).permute(1, 0, 2)
            ys = torch.full((src.size(0), 1), tgt_vocab["<pad>"], dtype=torch.long).to(device)
            for _ in range(20):  # generate max 20 tokens
                out = model_ws(src, ys)
                next_token = out[:, -1, :].argmax(-1).unsqueeze(1)
                ys = torch.cat([ys, next_token], dim=1)
            for seq in ys:
                decoded = [inv_tgt_vocab.get(tok.item(), "<unk>") for tok in seq if tok.item() not in [0, 1]]
                preds.append(" ".join(decoded))
            for label in batch["labels"]:
                decoded = [inv_tgt_vocab.get(tok.item(), "<unk>") for tok in label if tok.item() not in [0, 1]]
                refs.append([" ".join(decoded)])
    return bleu.compute(predictions=preds, references=refs)

# === Evaluate ===
print("\n==> Non-BPE Model BLEU Score:")
print(evaluate_bleu_ws())

# === Optional: Sample Sentence Test ===
print("\n==> Sample Translation (Non-BPE)")
test_sentence = "I saw Quazlor at the nightclub in Kreuzberg."
test_tokens = test_sentence.split()
src_ids = [src_vocab.get(t, src_vocab["<unk>"]) for t in test_tokens]
src_tensor = torch.tensor(src_ids + [0]*(64-len(src_ids))).unsqueeze(0).to(device)
ys = torch.full((1, 1), tgt_vocab["<pad>"], dtype=torch.long).to(device)
for _ in range(20):
    out = model_ws(src_tensor, ys)
    next_token = out[:, -1, :].argmax(-1).unsqueeze(1)
    ys = torch.cat([ys, next_token], dim=1)
decoded = [inv_tgt_vocab.get(tok.item(), "<unk>") for tok in ys[0] if tok.item() not in [0, 1]]
print("Translation:", " ".join(decoded))


Epoch 1/8:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2/8:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3/8:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4/8:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5/8:   0%|          | 0/50 [00:00<?, ?it/s]

...

Epoch 7/8:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 8/8:   0%|          | 0/50 [00:00<?, ?it/s]


==> Non-BPE Model BLEU Score:


Eval Non-BPE:   0%|          | 0/13 [00:00<?, ?it/s]

{'score': 0.2975928234249097, 'counts': [57, 2, 1, 0], 'totals': [1000, 950, 900, 850], 'precisions': [5.7, 0.21052631578947367, 0.1111111111111111, 0.058823529411764705], 'bp': 1.0, 'sys_len': 1000, 'ref_len': 850}

==> Sample Translation (Non-BPE)
Translation: in in in in in in in in in in in in in in in in in in in in


## Lecture 02: Monolingual Data

### Question: Why is monolingual data easier to obtain than parallel data for machine translation?

Monolingual data is easier to obtain than parallel data for machine translation because it exists naturally in far greater quantities and does not require specialized annotation or alignment. 

**Key reasons include:**

- **Abundance**: Any content written in a single language—such as books, news articles, blogs, or social media—qualifies as monolingual data. This content is produced organically across the internet and in print without the need for translation.
  
- **No alignment required**: Parallel data consists of sentence-aligned translations between two languages, which are expensive and time-consuming to create. In contrast, monolingual data does not require alignment or bilingual expertise.

- **Domain flexibility**: For specialized domains (e.g., legal, medical, or scientific), it is much more feasible to find large corpora in one language than to find aligned translations for both.

- **Resource availability**: For many low-resource languages or niche domains, parallel corpora may not exist at all, whereas monolingual resources (like news archives or public forums) are often still accessible.

As a result, monolingual data provides a valuable, scalable resource for improving machine translation systems—especially when parallel data is limited or costly to produce.


### Question: What key role did monolingual data play in statistical machine translation (SMT)?

In Statistical Machine Translation (SMT), monolingual data played a critical role in building the **language model**, which was one of the core components of the SMT architecture.

**Key contributions of monolingual data in SMT:**

- **Fluency modeling**: The language model, trained solely on monolingual target-language data, was responsible for ensuring that the generated translations were grammatically correct and fluent. It guided the decoder to prefer more natural-sounding sentence structures.

- **Complementing the translation model**: While the translation model was trained on parallel corpora to learn phrase-to-phrase alignments and mappings, the language model added an independent signal to prioritize well-formed target-language output.

- **Handling sparsity in parallel data**: Monolingual corpora were typically much larger and more diverse than available parallel data, allowing the language model to generalize better to unseen or rare target-language sequences.

- **Domain adaptation**: Domain-specific monolingual data helped tune the SMT system to generate fluent translations in specific fields (e.g., medical or legal), even when parallel data in those domains was scarce.

In essence, the language model in SMT—built from monolingual data—helped balance translation adequacy (what to say) with fluency (how to say it).


### Question: What is the difference between parallel and serial integration of a language model into an NMT decoder?

The difference between **parallel** and **serial** integration of a language model into a Neural Machine Translation (NMT) decoder lies in how the language model and decoder interact during the prediction of the next target word.

---

**Parallel Integration:**
- Both the **NMT decoder** and the **language model** independently process the previous target word embedding.
- Each model generates its own hidden state (e.g., `s_t` from the decoder and `s*_t` from the language model).
- These hidden states are then **combined (e.g., via averaging or concatenation)** to predict the next target word.
- Both models operate side-by-side and contribute equally to the final output layer.

**Advantages**:
- Maintains independence between models.
- Easy to pretrain each component separately.

**Limitations**:
- May miss deeper contextual blending of linguistic information.

---

***Serial Integration:***
- The **language model processes the previous target word embedding first**, producing a context-enriched hidden state (`s*_t`).
- This hidden state is then used as input to the **NMT decoder**, instead of the raw embedding.
- The decoder thus benefits from a more context-aware representation, potentially improving fluency.

**Advantages**:
- Provides the decoder with richer contextual signals.
- Can improve target-side fluency by incorporating long-range dependencies early.

**Limitations**:
- Tighter coupling may complicate training and optimization.
- Requires integration-aware model design.

---

In summary, **parallel integration** treats both components as peers, while **serial integration** uses the language model as a preprocessing step that enriches input to the decoder.


### Question: Why is combining a language model with a decoder considered a good way to use monolingual data?

Combining a language model with a decoder is considered a good way to use monolingual data because it allows the translation system to benefit from the fluency and context modeling capabilities of a language model without requiring changes to the core parallel training data pipeline.

This approach takes advantage of the structural similarity between the NMT decoder and a standalone language model. While the decoder in NMT uses both the source sentence and previously generated target words, a language model is trained solely on monolingual target-language sequences. By integrating a pre-trained language model into the decoder, the system can:

- Leverage large-scale monolingual data to improve fluency and grammaticality of translations.
- Incorporate contextual information from the target language that may not be well represented in limited parallel data.
- Improve translation quality in low-resource scenarios by supplementing sparse target-language signals.
- Enable flexible training strategies such as pretraining, joint training, or multi-task learning.

This method allows NMT systems to generalize better in real-world settings where monolingual data is abundant but parallel data is limited.


### Question: What are the main trade-offs between pretraining a language model and training it jointly with the NMT decoder?

The main trade-offs between pretraining a language model and training it jointly with the NMT decoder involve differences in control, efficiency, flexibility, and final translation quality.

**Pretraining a language model (then fine-tuning with the NMT decoder):**

- **Advantages**:
  - Enables the use of large-scale monolingual corpora before any parallel data is involved.
  - Allows for more focused optimization of the language model on fluency and contextual representation.
  - Reduces training complexity during the NMT phase since the language model is already trained.

- **Disadvantages**:
  - May require careful architectural alignment between the pretrained model and the NMT decoder.
  - Risk of forgetting the pretrained fluency benefits during fine-tuning on parallel data (catastrophic forgetting).
  - Fixed integration may limit adaptability during NMT training.

**Joint training (multi-task learning with alternating monolingual and parallel updates):**

- **Advantages**:
  - Allows the model to continuously reinforce both fluency (from monolingual data) and translation accuracy (from parallel data).
  - Supports more flexible and dynamic learning of both objectives in a shared representation space.
  - Avoids alignment issues by co-training all components together.

- **Disadvantages**:
  - More complex training setup and scheduling logic (e.g., how often to alternate between tasks).
  - Higher memory and computational cost due to the combined training process.
  - Risk of one task dominating the learning process if not carefully balanced.

In summary, pretraining provides modularity and simplicity, while joint training offers integrated learning but at a higher cost and complexity.


### Question: Why is it generally better to have errors on the source side of synthetic data than on the target side?

It is generally better to have errors on the source side of synthetic data than on the target side because the quality of the target sentence directly influences what the model learns to generate. In machine translation, the target side serves as the reference output, guiding the model's understanding of correct grammar, fluency, and meaning in the target language.

If the target side contains errors, the model will learn to imitate incorrect or unnatural translations, reducing overall translation quality. These errors propagate directly into the output and can degrade fluency and accuracy.

In contrast, errors on the source side are less harmful because the model is trained to map potentially noisy input to a clean, high-quality output. Since real-world inputs may be noisy or ungrammatical, training on imperfect source data can even improve the model’s robustness. The model still benefits from learning how to produce correct target-language sentences, which is the ultimate goal in translation.

Therefore, preserving high-quality, human-authored target sentences ensures the model learns proper output patterns, even if the corresponding source inputs are machine-generated or partially inaccurate.


### Question: What is backtranslation and in which scenarios would backtranslation be preferable to integrating a language model?

Backtranslation is a data augmentation technique used in Neural Machine Translation (NMT) where monolingual target-language data is translated into the source language using a reverse translation model. This process creates synthetic parallel data, which is then used to train or fine-tune the original translation model. For example, to improve a German-to-English model, one might use an English-to-German model to generate synthetic German sentences from monolingual English text.

Backtranslation is preferable to integrating a language model in scenarios such as:

- **Low-resource parallel data**: When there is very limited bilingual data, backtranslation enables the use of abundant monolingual target-language data to expand the training set.
- **Architectural simplicity**: Backtranslation does not require modifications to the NMT architecture, unlike language model integration, which involves architectural design choices and training coordination.
- **Domain adaptation**: When adapting a general model to a specific domain (e.g., legal or medical), monolingual domain-specific text can be backtranslated to produce in-domain synthetic parallel data.
- **Existing reverse model availability**: If a reasonably accurate reverse-direction translation model is already available, backtranslation is easy to implement and scales well.
- **Focus on output fluency**: Because the target side of the synthetic data remains high-quality (human-written), backtranslation helps maintain strong output fluency and naturalness.

Overall, backtranslation is a practical and effective method for leveraging monolingual data, especially when clean target-language text is easier to obtain than source-language translations.


### Example: BackTranslation

In [22]:
# Byte Pair Encoding vs Word-Based NMT Example
# Goal: Train a translation model using Byte Pair Encoding (BPE) to highlight its impact on rare words/named entities.
# We'll use Hugging Face datasets and tokenizers. No GPU required, but it will run faster if one is available.

# Step 1: Setup and imports
from datasets import load_dataset, Dataset
from transformers import MarianMTModel, MarianTokenizer
import torch

# Ensure we're using GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 2: Load monolingual data (target language: English)
# We'll simulate a small monolingual dataset of English sentences
monolingual_en_sentences = [
    {"en": "The meeting was postponed due to rain."},
    {"en": "A new species of bird was discovered in the Amazon."},
    {"en": "Backtranslation helps improve machine translation quality."},
    {"en": "She loves to cook Italian food on weekends."},
    {"en": "The software update includes several new features."},
    {"en": "For some reason, our Data Scientists were not enformed of the new features."}

]

# Wrap into a Hugging Face Dataset for easy processing
mono_en_dataset = Dataset.from_list(monolingual_en_sentences)

# Step 3: Load the reverse translation model (English → German)
reverse_model_name = "Helsinki-NLP/opus-mt-en-de"
reverse_tokenizer = MarianTokenizer.from_pretrained(reverse_model_name)
reverse_model = MarianMTModel.from_pretrained(reverse_model_name).to(device)

# Step 4: Translate English sentences to synthetic German (backtranslation source)
def translate_to_german(example):
    # Tokenize input English sentence
    inputs = reverse_tokenizer(example["en"], return_tensors="pt", padding=True, truncation=True).to(device)
    # Generate translation (English → German)
    translated = reverse_model.generate(**inputs, max_length=64)
    # Decode the translation
    example["de"] = reverse_tokenizer.decode(translated[0], skip_special_tokens=True)
    return example

# Apply the translation function to each example
synthetic_parallel = mono_en_dataset.map(translate_to_german)

# Step 5: Display the resulting synthetic parallel dataset
for example in synthetic_parallel:
    print(f"EN: {example['en']}")
    print(f"DE: {example['de']}")
    print("---")

# Notes:
# - In real training, you would now use this synthetic (DE, EN) data to train a DE→EN model.
# - This example only shows the first stage of backtranslation.
# - You can repeat the same logic in reverse for target→source if needed.
# - For larger datasets, batching and efficient tokenization are recommended.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

EN: The meeting was postponed due to rain.
DE: Das Treffen wurde wegen Regens verschoben.
---
EN: A new species of bird was discovered in the Amazon.
DE: Im Amazonas wurde eine neue Vogelart entdeckt.
---
EN: Backtranslation helps improve machine translation quality.
DE: Backtranslation hilft, die Qualität der maschinellen Übersetzung zu verbessern.
---
EN: She loves to cook Italian food on weekends.
DE: Sie kocht am Wochenende gerne italienisches Essen.
---
EN: The software update includes several new features.
DE: Das Software-Update enthält mehrere neue Funktionen.
---
EN: For some reason, our Data Scientists were not enformed of the new features.
DE: Aus irgendeinem Grund waren unsere Data Scientists nicht über die neuen Funktionen informiert.
---


### Question: Describe how you would train an NMT system using only a small amount of parallel data and a large monolingual corpus.

To train a Neural Machine Translation (NMT) system using only a small amount of parallel data and a large monolingual corpus, you can follow a multi-step approach that combines initial supervised training with synthetic data generation:

1. **Train an initial model with the small parallel dataset**:
   - Use the available parallel data (e.g., German–English) to train a baseline NMT model in the reverse direction (e.g., English→German).
   - This reverse model will be used for backtranslation.

2. **Generate synthetic source sentences via backtranslation**:
   - Use the trained reverse model to translate a large monolingual corpus from the target language (e.g., English) into the source language (e.g., German).
   - This creates synthetic parallel sentence pairs where the source side is machine-generated and the target side is high-quality monolingual data.

3. **Combine original and synthetic data**:
   - Merge the synthetic parallel data with the original small parallel dataset.
   - This expanded dataset provides more coverage of vocabulary, style, and domain-specific usage.

4. **Train the main NMT model (source→target)**:
   - Use the combined dataset to train the final model in the desired direction (e.g., German→English).
   - Optionally use techniques like curriculum learning or noise-aware training to handle the lower quality of the synthetic source data.

5. **(Optional) Fine-tune on high-quality parallel data**:
   - After training on the large mixed dataset, fine-tune the model on the original high-quality parallel data to recover any lost precision.

This strategy allows the NMT model to benefit from both the fluency of monolingual target data and the translation mappings learned from limited parallel examples.


### Example: NMT system

In [24]:
# Step-by-step backtranslation and training preparation using Hugging Face
# Goal: Simulate how to train an NMT system using limited parallel data and a large monolingual corpus

from datasets import load_dataset, Dataset
from transformers import MarianMTModel, MarianTokenizer
import torch

# 1. Load small parallel dataset (simulate low-resource parallel data)
parallel_data = load_dataset("opus_books", "de-en", split="train[:1%]")

# 2. Load monolingual English data (simulate large monolingual corpus)
# For demonstration, extract English text from another slice of the same dataset
monolingual_raw = load_dataset("opus_books", "de-en", split="train[1%:2%]")
monolingual_data = [ex["translation"]["en"] for ex in monolingual_raw]

# 3. Load reverse translation model: English → German
reverse_model_name = "Helsinki-NLP/opus-mt-en-de"
reverse_tokenizer = MarianTokenizer.from_pretrained(reverse_model_name)
reverse_model = MarianMTModel.from_pretrained(reverse_model_name)

# Move model to available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reverse_model.to(device)

# 4. Translate monolingual English to synthetic German
def backtranslate_batch(sentences, tokenizer, model, device, batch_size=8):
    synthetic_pairs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
        translated = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        synthetic_pairs.extend(zip(translated, batch))
    return synthetic_pairs

synthetic_parallel = backtranslate_batch(monolingual_data, reverse_tokenizer, reverse_model, device)

# 5. Combine synthetic data with real parallel data
combined_data = []

# Add original parallel data
for ex in parallel_data:
    combined_data.append({
        "de": ex["translation"]["de"],
        "en": ex["translation"]["en"]
    })

# Add synthetic parallel data
for de, en in synthetic_parallel:
    combined_data.append({
        "de": de,
        "en": en
    })

# Wrap into Hugging Face Dataset object
combined_dataset = Dataset.from_list(combined_data)

# 6. Preview a few examples from the combined dataset
import pandas as pd
from IPython.display import display

df_preview = pd.DataFrame(combined_dataset[:10])
display(df_preview)


,de,en
0,Source: http://www.zeno.org - Contumax GmbH & ...,Source: Project Gutenberg
1,Jane Eyre,Jane Eyre
2,Charlotte Bronte,Charlotte Bronte
3,Erstes Kapitel,CHAPTER I
4,"Es war ganz unmöglich, an diesem Tage einen Sp...",There was no possibility of taking a walk that...
5,Am Morgen waren wir allerdings während einer g...,"We had been wandering, indeed, in the leafless..."
6,Ich war von Herzen froh darüber: lange Spazier...,"I was glad of it: I never liked long walks, es..."
7,"Die soeben erwähnten Eliza, John und Georgina ...","The said Eliza, John, and Georgiana were now c..."
8,"Mich hatte sie davon dispensiert, mich der Gru...","Me, she had dispensed from joining the group; ..."
9,"»Was sagt denn Bessie, daß ich gethan habe?« f...","""What does Bessie say I have done?"" I asked."


## Lecture 03: Multilingual MT

### Question: Are multilingual models have better preformance compared to specialized models? In other words, do models that are exposed to many languages preform better at translating German, for instance, compared to models that have only been exposed to language pairs english to german? 

Multilingual neural machine translation (MNMT) models can outperform bilingual models in certain contexts, especially when translating between high-resource language pairs like English and German. However, the relative performance of multilingual versus bilingual systems depends on several factors, including data availability, language similarity, and training strategy.

**Advantages of multilingual models:**

- **Improved performance for high-resource languages:** Multilingual models trained on a broad set of language pairs have demonstrated competitive or superior performance on high-resource pairs. For example, the original Google MNMT system outperformed strong bilingual baselines on English↔German in the WMT’14 benchmark (Johnson et al., 2017).  
  *Source: [Johnson et al., 2017, arXiv:1611.04558](https://arxiv.org/abs/1611.04558)*

- **Zero-shot translation:** Multilingual systems can perform translations between language pairs not seen during training, leveraging shared representations learned across multiple languages.  
  *Source: [Johnson et al., 2017, arXiv:1611.04558](https://arxiv.org/abs/1611.04558)*

- **Parameter sharing and efficiency:** Training a single model to handle many language pairs reduces the need to maintain multiple systems and can improve generalization through shared representations.

**Challenges and trade-offs:**

- **Low-resource language performance:** For some low-resource language pairs, bilingual models trained solely on that pair may still outperform multilingual systems due to focused capacity. Multilingual models may underfit these pairs if high-resource languages dominate training.  
  *Source: [Bapna et al., 2024, Findings of NAACL](https://aclanthology.org/2024.findings-naacl.176.pdf)*

- **Balancing training data:** In multilingual settings, care must be taken to avoid overrepresenting high-resource languages during training, which can bias the model and reduce performance on underrepresented pairs.

- **Domain-specific applications:** Bilingual models fine-tuned for a particular domain (e.g., medical, legal) often yield better in-domain performance compared to general multilingual models trained across broad domains.

  *Source:[arXiv:2407.14076](https://arxiv.org/abs/2407.14076)*

**Conclusion:**

Multilingual models offer strong advantages in scalability, cross-lingual generalization, and zero-shot capabilities. For well-resourced language pairs like English↔German, multilingual systems often match or exceed the performance of specialized bilingual models. However, in low-resource or domain-specific settings, dedicated bilingual systems may still be preferable. The choice should be guided by the specific requirements of the task, the availability of data, and whether multi-domain or cross-lingual capabilities are needed.


## Question: What is a pivot-based translation system and what is the main drawback of pivot-based translation systems that use a third language like English as an intermediary?

A pivot-based translation system is a method used to translate between two languages that do not have direct parallel data by using a third, intermediate language (called the pivot). This is common in multilingual settings where most parallel data is available between other languages and a central language like English.

For example, to translate from Maltese to Lithuanian:
1. First translate Maltese → English (pivot)
2. Then translate English → Lithuanian

**Main drawback of pivot-based translation:**

- **Accumulation of errors**: Each translation step introduces potential inaccuracies. Errors from the first step (source → pivot) can propagate and be amplified in the second step (pivot → target).

- **Ambiguity in the pivot language**: The pivot language (often English) may have ambiguous words or structures that lose meaning or precision during intermediate translation.

- **Increased latency**: Translating in two steps rather than one increases computational time and resource use.

- **Loss of cultural or structural nuances**: Some expressions or syntax may not map well into the pivot language, leading to degraded translation quality in the final output.


### Question: How does multitask learning enable multilingual NMT systems to perform zero-shot translation?

Multitask learning enables multilingual Neural Machine Translation (NMT) systems to perform zero-shot translation by training the model on multiple translation tasks simultaneously across different language pairs. This approach helps the model learn shared representations that generalize beyond the specific language pairs seen during training.

**How it works:**

- **Shared encoder-decoder architecture**: The model uses a common encoder and decoder across many language pairs, which encourages the learning of language-independent representations.

- **Cross-lingual transfer**: Training on diverse language pairs (e.g., English↔German, English↔Chinese, English↔Korean) allows the model to map semantically similar inputs to similar internal representations, regardless of language.

- **Zero-shot inference**: At test time, the model can translate between language pairs it has never seen directly (e.g., Chinese→German), because it has learned to represent concepts in a shared latent space through its exposure to related language pairs.

**Key enablers:**
- Shared vocabulary or multilingual embeddings
- Language tokens or annotations to indicate the target language
- Balanced training across language pairs to prevent dominance by high-resource pairs

By learning to generalize across translation tasks, multitask-trained models can extend their capabilities to unseen translation directions without requiring new parallel data.


### Question: What are the main architectural differences between a multilingual NMT model with separate encoders/decoders and one with shared components?

The main architectural differences between a multilingual NMT model with separate encoders/decoders and one with shared components relate to how language-specific versus language-agnostic processing is handled.

**Multilingual model with separate encoders/decoders:**

- Each source language has its own encoder.
- Each target language has its own decoder.
- A shared attention mechanism may be used to link encoders and decoders.
- Language-specific parameters allow the model to specialize in the syntax and semantics of each language.
- Only the relevant encoder/decoder pair is updated during training on a specific language pair.
- Greater model size and complexity as the number of languages increases.

**Multilingual model with shared components:**

- A single encoder is used for all source languages.
- A single decoder is used for all target languages.
- A shared attention mechanism connects encoder and decoder across all tasks.
- Tokens may be tagged with language identifiers to disambiguate meaning (e.g., `de_Haus` vs `en_house`).
- Shared parameters promote generalization and enable zero-shot translation between unseen language pairs.
- More compact and scalable model design.

In summary, separate-component architectures prioritize specialization and control, while shared-component architectures emphasize generalization and parameter efficiency.


### Question: Why is it helpful to add language-specific tags or tokens to the vocabulary in multilingual translation systems?

Adding language-specific tags or tokens to the vocabulary in multilingual translation systems helps the model disambiguate between words that appear similar across different languages but have different meanings, roles, or grammatical behaviors.

**Reasons this is helpful:**

- **Vocabulary overlap resolution**: Some words (e.g., "die") exist in multiple languages but have different meanings. Without tags, the model may incorrectly share embeddings or assign inappropriate context.
  - Example: "die" in English (verb) vs. "die" in German (feminine article).

- **Improved token-level clarity**: By prefixing tokens with a language identifier (e.g., `de_Haus`, `en_house`), the model can learn separate embeddings for each language, even when the surface forms are identical.

- **Better context handling**: Language tags help the model infer the syntactic and semantic context specific to each language, which supports better alignment and generation in multilingual settings.

- **Support for zero-shot translation**: Explicit language cues help guide the decoder when generating output in a target language it has not seen in training for a given source pair.

- **Scalable representation**: This technique allows a single shared model to handle dozens or hundreds of languages without significant interference, especially when combined with subword tokenization.

Overall, language-specific tokens provide a simple but effective mechanism to improve multilingual model performance by reducing confusion and enabling clearer language separation in the shared embedding space.


### Example: Langugage-Specific Taggin

In [25]:
# Simulating language-specific token tagging in a multilingual NMT preprocessing pipeline

# Sample multilingual data
examples = [
    {"lang": "en", "text": "The house is beautiful."},
    {"lang": "de", "text": "Das Haus ist schön."},
    {"lang": "fr", "text": "La maison est belle."}
]

# Language-specific tagging function
def tag_text_with_lang(example):
    lang_tag = f"<{example['lang']}>"
    # Prefix the sentence with a language token
    return f"{lang_tag} {example['text']}"

# Apply tagging to each example
tagged_sentences = [tag_text_with_lang(ex) for ex in examples]

# Display the tagged output
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.DataFrame({
    "original": [ex["text"] for ex in examples],
    "tagged": tagged_sentences,
    "language": [ex["lang"] for ex in examples]
})

display(df)


,original,tagged,language
0,The house is beautiful.,<en> The house is beautiful.,en
1,Das Haus ist schön.,<de> Das Haus ist schön.,de
2,La maison est belle.,<fr> La maison est belle.,fr


### Question: How can the concept of a shared, language-independent representation be extended to modalities like images or audio in NMT systems?

The concept of a shared, language-independent representation can be extended to modalities like images or audio in Neural Machine Translation (NMT) systems by using encoders that project these non-text inputs into the same semantic space used for text. This allows the model to perform tasks such as image captioning or speech translation within the same architecture used for multilingual text translation.

**How this works:**

- **Unified encoder space**: Non-text inputs (e.g., images or audio) are passed through modality-specific encoders (e.g., CNNs for images, speech models for audio) to produce vector representations that capture the input’s semantic content.

- **Shared decoder**: A common decoder, often used in text-based translation, can then be used to generate outputs in the target language using these representations.

- **Multimodal learning**: By training the model on multiple types of input-output pairs (e.g., image→caption, speech→transcript, English→German), it learns to map all inputs into a shared latent space that is semantically rich and language-agnostic.

- **Zero-shot generalization**: Once this space is learned, the system may be able to generate captions or translations in languages it hasn't seen directly paired with a specific modality (e.g., generate a German caption from an image despite training only on English captions).

- **Applications**:
  - Image captioning (image → text)
  - Speech transcription (audio → same-language text)
  - End-to-end speech translation (audio → target-language text)

This shared representation framework enables NMT systems to flexibly handle diverse input types and scale across both languages and modalities.


## Lecture the last

### Rant

This lecture should not be the final one in the series—it should be the *starting point*. Introducing the Transformer architecture as a sort of epilogue to a long parade of LSTM-based architectures fundamentally misrepresents the current state and future of machine translation.

**Reasons this is a poor placement in the sequence:**

- **Transformers define the modern era of NMT**: Nearly every competitive model in the field today—whether in translation, summarization, question answering, or speech—is built on top of the Transformer architecture. Positioning it last gives the impression that it's an optional or "advanced" topic, rather than the baseline from which all new models evolve.

- **The self-attention mechanism reframes the problem**: Unlike LSTMs, which struggle with long-range dependencies and rely on sequential processing, self-attention gives every token global access to the sequence from the start. This isn't just an efficiency boost—it's a paradigm shift.

- **All the current tools build from here**: Models like BERT, GPT, T5, and Whisper use transformers as their core. Any serious machine translation engineer or NLP researcher today will be working with these architectures, not LSTMs. Burying transformers at the end of the series undercuts the relevance of the material that comes before.

- **Pedagogically backwards**: Students deserve to understand the tools that actually define the field *before* they spend hours wrestling with architectures that have mostly been superseded. A course structured around the evolution of attention mechanisms—with RNNs as historical context—would be far more valuable.

In short, this lecture is not a capstone—it's the foundation. Treating it as anything else risks leaving students with the wrong conceptual map of the field.


### Question: Why does the Transformer architecture represent a fundamental shift in how we model sequences compared to LSTM-based models?

The Transformer architecture represents a fundamental shift in how we model sequences because it abandons recurrence entirely. Instead of processing tokens one at a time and passing state forward like an LSTM, Transformers let every token *see* the entire sequence at once using self-attention. This change unlocks several major differences:

**1. Global context, instantly:**
- In LSTMs, each token has limited context—what came before (or after, in bidirectional models).
- Transformers allow each word to directly attend to *every other* word in the sequence, making context global rather than sequential.

**2. Parallelization becomes possible:**
- LSTMs process sequences step-by-step, making training inherently slow and difficult to scale.
- Transformers process all tokens simultaneously, which makes them far more efficient and GPU-friendly.

**3. Long-range dependencies are no longer a bottleneck:**
- LSTMs struggle to retain and propagate information across long sequences due to vanishing gradients and capacity limits.
- Transformers can model long-range dependencies with a single attention hop, regardless of sentence length.

**4. Attention becomes the central unit of computation:**
- Instead of encoding everything into a fixed-size vector or hidden state, Transformers calculate pairwise interactions between tokens through dot-product attention.
- This not only improves interpretability, it shifts the inductive bias away from memory and recurrence toward pattern-matching and alignment.

**5. Position is now explicit, not implicit:**
- LSTMs rely on order being "baked in" via recurrence.
- Transformers use positional encodings to inject information about token order, giving the model more flexibility in how it reasons about sequence structure.

In short, Transformers reframed sequence modeling from a left-to-right memory game into a pattern-rich space of relationships. It’s not a minor upgrade—it’s a complete rewrite of how sequence understanding works.


### Question: What is multi-head attention and how does multi-head attention improve on single-head attention, and what does this imply about the Transformer’s ability to model different aspects of language?

Multi-head attention is a technique in the Transformer architecture where the model computes several attention mechanisms in parallel—each with its own set of learned linear projections for queries, keys, and values. Instead of producing a single attention output per token, multi-head attention generates multiple attention outputs (or “heads”) that are then concatenated and linearly transformed into a final representation.

**Why it improves on single-head attention:**

- **Diverse focus:** Each head can focus on different parts of the sequence or learn different relational patterns—one might focus on syntactic roles, another on semantic similarity, another on positional structure.
- **Subspace specialization:** Since each head operates in a lower-dimensional space (a fraction of the model’s full hidden size), it encourages specialization and diversity of attention patterns.
- **Rich compositionality:** By combining the outputs from multiple perspectives, the model gains a more nuanced and layered understanding of the sentence.

**What this implies about the Transformer's ability to model language:**

- Transformers don't just look at *where* a token should attend, but at *how many different ways* that attention can be meaningful.
- It allows the model to simulate multiple kinds of linguistic analysis simultaneously—dependency structure, coreference, tense agreement, discourse flow, etc.—without being told explicitly to do so.
- The attention mechanism becomes a multi-threaded workspace for understanding language, enabling generalization across varied tasks and linguistic phenomena.

In essence, multi-head attention lets the model see language through many lenses at once—and it turns out, that’s exactly what you need to make sense of it.


### Current Questions

The following are some of the current cutting-edge research questions in machine translation (MT), along with recent scientific papers that address them:

**1. How can we improve translation quality for low-resource languages?**

- Many languages lack sufficient parallel corpora, making it difficult to train effective MT systems.

- Relevant papers:
  - [Understanding and Analyzing Model Robustness and Knowledge-Transfer in Multilingual Neural Machine Translation using TX-Ray](https://arxiv.org/abs/2412.13881)
  - [Joint speech and text machine translation for up to 100 languages (SEAMLESSM4T)](https://www.nature.com/articles/s41586-024-08359-z)

**2. How can we effectively translate complex and culturally nuanced content, such as literary texts?**

- Literary translation requires deep understanding of figurative language, cultural context, and authorial style.

- Relevant papers:
  - [Perhaps Beyond Human Translation: Harnessing Multi-Agent Collaboration for Translating Ultra-Long Literary Texts](https://arxiv.org/abs/2405.11804)
  - [Towards Cross-Cultural Machine Translation with Retrieval-Augmented Models](https://machinelearning.apple.com/research/cultural-translation)

**3. How can we ensure the robustness and reliability of MT systems in real-world applications?**

- MT systems should handle diverse and noisy input and maintain performance across domains.

- Relevant papers:
  - [Understanding and Analyzing Model Robustness and Knowledge-Transfer in Multilingual Neural Machine Translation using TX-Ray](https://arxiv.org/abs/2412.13881)
  - [Salute the Classic: Revisiting Challenges of Machine Translation in the Age of LLMs](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00730/127458/Salute-the-Classic-Revisiting-Challenges-of)

**4. How can we integrate multimodal data (e.g., images, audio) into MT systems?**

- Multimodal input enhances translation in contexts where text alone is insufficient.

- Relevant papers:
  - [Joint speech and text machine translation for up to 100 languages (SEAMLESSM4T)](https://www.nature.com/articles/s41586-024-08359-z)
  - [M3T: A new benchmark dataset for multi-modal document-level machine translation](https://www.amazon.science/tag/machine-translation)

**5. How can we adapt MT systems to specific domains and user preferences?**

- Domain-specific adaptation is crucial for technical, legal, scientific, and personalized translation tasks.

- Relevant papers:
  - [Adaptive machine translation](https://en.wikipedia.org/wiki/Adaptive_machine_translation)
  - [Science Across Languages: Assessing LLM Multilingual Translation of Scientific Papers](https://arxiv.org/abs/2502.17882)

These questions represent the forefront of modern MT research, especially as models scale to new languages, modalities, and application contexts.
